# Whisper v3 is here!

Whisper v3 is a new model open sourced by OpenAI. The model can do multilingual transcriptions and is quite impressive. For example, you can change from English to Spanish or Chinese in the middle of a sentence and it will work well!

The model can be run in a free Google Colab instance and is integrated into transformers already, so switching can be a very smooth process if you already use the previous versions.

In [1]:
!pip install git+https://github.com/huggingface/transformers gradio

  Cloning https://github.com/huggingface/transformers to /private/var/folders/36/93s3hflx3zsgybvbrpldfvym0000gn/T/pip-req-build-malkmr_o
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers /private/var/folders/36/93s3hflx3zsgybvbrpldfvym0000gn/T/pip-req-build-malkmr_o
  Resolved https://github.com/huggingface/transformers to commit f0f456b365eaf0479faabc35c43d29bf6c230003
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of transformers to determine which version is compatible with other requirements. This could take a while.
ERROR: Ignored the following yanked versions: 0.8.0, 0.9.0.dev0, 0.9.0rc0, 0.16.1, 0.26.4, 0.31.3
ERROR: Could not find a version that satisfies the requirement huggingface-hub<2.0,>=1.5.0 (from transformers) (from versions: 0.0.1, 0.0.2, 0.0.3rc1, 0.0.3rc2, 0.0.5, 0.0.6, 0.0.7, 0.0.8, 0.0.9, 

In [2]:
import sys
# Désinstallation pour éviter les conflits de fichiers corrompus
!{sys.executable} -m pip uninstall -y transformers accelerate gradio numpy
# Installation des versions stables pour Python 3.9/Mac
!{sys.executable} -m pip install "numpy<2.0" "gradio==3.50.2" transformers accelerate

Found existing installation: transformers 4.57.6
Uninstalling transformers-4.57.6:
  Successfully uninstalled transformers-4.57.6
Found existing installation: accelerate 1.10.1
Uninstalling accelerate-1.10.1:
  Successfully uninstalled accelerate-1.10.1
Found existing installation: gradio 3.50.2
Uninstalling gradio-3.50.2:
  Successfully uninstalled gradio-3.50.2
Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
  Using cached numpy-1.26.4-cp39-cp39-macosx_10_9_x86_64.whl (20.6 MB)
  Using cached gradio-3.50.2-py3-none-any.whl (20.3 MB)
  Using cached transformers-4.57.6-py3-none-any.whl (12.0 MB)
  Using cached accelerate-1.10.1-py3-none-any.whl (374 kB)
You should consider upgrading via the '/usr/local/bin/python3.9 -m pip install --upgrade pip' command.


Let's use the high level pipeline from the transformers library to load the model.

{'text': " going along slushy country roads and speaking to damp audiences in draughty schoolrooms day after day for a fortnight he'll have to put in an appearance at some place of worship on sunday morning and he can come to us immediately afterwards"}
Let's now build a quick Gradio demo where we can play with the model directly using our microphone! You can run this code in a Google Colab instance (or locally!) or just head to the Space to play directly with it online.

Setting queue=True in a Colab notebook requires sharing enabled. Setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Running on public URL: https://037dbdb04542aa1a29.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)

In [3]:
import torch
from transformers import pipeline

# Configuration matérielle Mac
device = "mps" if torch.backends.mps.is_available() else "cpu"
torch_dtype = torch.float16 if device == "mps" else torch.float32

print(f"Chargement de Whisper v3 sur : {device}")

# On charge le modèle v3
# À remplacer dans la cellule de chargement du modèle
pipe = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-large-v3",
    torch_dtype=torch.float32, # Plus stable sur tous les Mac
    device=device
)

Chargement de Whisper v3 sur : cpu


Device set to use cpu


In [4]:
import gradio as gr

def transcribe(audio):
    if audio is None:
        return "Veuillez enregistrer ou charger un fichier audio."
    
    # Transcription avec Whisper v3
    result = pipe(audio, generate_kwargs={"task": "transcribe"})
    return result["text"]

# Création de l'interface
demo = gr.Interface(
    fn=transcribe,
    inputs=gr.Audio(source="microphone", type="filepath", label="Enregistrer votre voix"),
    outputs=gr.Textbox(label="Texte transcrit"),
    title="Test Whisper v3 - Speech to Text",
    description="Parlez dans le micro et attendez la transcription (optimisé pour Mac)."
)

demo.launch()

Running on local URL:  http://127.0.0.1:7861

To create a public link, set `share=True` in `launch()`.


IMPORTANT: You are using gradio version 3.50.2, however version 4.44.1 is available, please upgrade.
--------


/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/site-packages/pydub/utils.py:198: RuntimeWarning: Couldn't find ffprobe or avprobe - defaulting to ffprobe, but may not work
  warn("Couldn't find ffprobe or avprobe - defaulting to ffprobe, but may not work", RuntimeWarning)
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/site-packages/gradio/processing_utils.py", line 145, in audio_from_file
    audio = AudioSegment.from_file(filename)
  File "/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/site-packages/pydub/audio_segment.py", line 728, in from_file
    info = mediainfo_json(orig_file, read_ahead_limit=read_ahead_limit)
  File "/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/site-packages/pydub/utils.py", line 274, in mediainfo_json
    res = Popen(command, stdin=stdin_parameter, stdout=PIPE, stderr=PIPE)
  File "/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/sub

In [5]:
import torch
from transformers import pipeline
import gradio as gr

# 1. On force le CPU si vous avez un doute, ou MPS pour puce M1/M2/M3
# ATTENTION : Sur Mac Intel, float16 provoque systématiquement une erreur.
device = "mps" if torch.backends.mps.is_available() else "cpu"
current_dtype = torch.float32 # On utilise float32 pour une compatibilité totale

print(f"Initialisation sur {device} avec dtype {current_dtype}")

# 2. Chargement du pipeline avec paramètres de sécurité
pipe = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-large-v3",
    torch_dtype=current_dtype,
    device=device
)

# 3. Fonction de transcription avec gestion d'erreur visible
def transcribe(audio):
    if audio is None:
        return "Aucun audio détecté. Vérifiez l'accès au micro."
    try:
        # On force la transcription
        result = pipe(audio, generate_kwargs={"task": "transcribe"})
        return result["text"]
    except Exception as e:
        # Ceci affichera la VRAIE erreur dans Gradio si ça plante
        return f"Erreur technique : {str(e)}"

# 4. Interface (Syntaxe stable pour Gradio 3.x)
demo = gr.Interface(
    fn=transcribe,
    inputs=gr.Audio(source="microphone", type="filepath"),
    outputs="text",
    title="Test Whisper v3"
)

demo.launch(debug=True) # debug=True permet de voir les erreurs dans le notebook

Initialisation sur cpu avec dtype torch.float32


Device set to use cpu


Running on local URL:  http://127.0.0.1:7862

To create a public link, set `share=True` in `launch()`.


IMPORTANT: You are using gradio version 3.50.2, however version 4.44.1 is available, please upgrade.
--------


/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/site-packages/pydub/utils.py:198: RuntimeWarning: Couldn't find ffprobe or avprobe - defaulting to ffprobe, but may not work
  warn("Couldn't find ffprobe or avprobe - defaulting to ffprobe, but may not work", RuntimeWarning)
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/site-packages/gradio/processing_utils.py", line 145, in audio_from_file
    audio = AudioSegment.from_file(filename)
  File "/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/site-packages/pydub/audio_segment.py", line 728, in from_file
    info = mediainfo_json(orig_file, read_ahead_limit=read_ahead_limit)
  File "/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/site-packages/pydub/utils.py", line 274, in mediainfo_json
    res = Popen(command, stdin=stdin_parameter, stdout=PIPE, stderr=PIPE)
  File "/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/sub

Keyboard interruption in main thread... closing server.
